# Olist: chọn thời điểm dự đoán, rồi đổi metric ra tiền

Bài toán: cảnh báo đơn có nguy cơ giao trễ để CSKH nhắn tin xin lỗi và báo ngày giao mới **trước khi**
đơn tới tay khách. Vì can thiệp trước khi giao, mọi thông tin sau thời điểm dự đoán (ngày giao thực tế,
review) đều là leak. Đây là điểm khác với model review xấu "lúc đơn vừa giao" của module 10/11.

Hai thời điểm ứng viên:
- **purchase**: khách vừa đặt hàng.
- **carrier**: seller vừa giao đơn cho đơn vị vận chuyển (biết thêm số ngày seller xử lý, số ngày còn lại tới ngày hẹn).

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore`) đọc dữ liệu thật, gọi hàm, vẽ. Test bỏ qua chúng.
- Chấm: `uv run pytest tracks/data-science/modules/15-ml-system-design/exercises/intermediate/01-olist-decision-time -q`

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import average_precision_score, roc_auc_score

In [ ]:
# explore: đọc Olist thật (tải bằng: uv run python data/download_olist.py)
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data/raw"
dates = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
         "order_delivered_customer_date", "order_estimated_delivery_date"]
orders = pd.read_csv(RAW / "olist_orders_dataset.csv", parse_dates=dates)
items = pd.read_csv(RAW / "olist_order_items_dataset.csv")
customers = pd.read_csv(RAW / "olist_customers_dataset.csv")
sellers = pd.read_csv(RAW / "olist_sellers_dataset.csv")
orders.shape, items.shape

## 1. Bảng một dòng mỗi đơn

Mỗi feature phải trả lời được câu "lúc dự đoán đã có giá trị này chưa?". `handling_days` và
`remaining_days` chỉ có từ lúc giao cho vận chuyển.

In [ ]:
def build_orders(orders: pd.DataFrame, items: pd.DataFrame, customers: pd.DataFrame,
                 sellers: pd.DataFrame) -> pd.DataFrame:
    """Một dòng mỗi đơn giao xong (delivered + có order_delivered_customer_date).

    Cột: order_id, purchase_ts, late (int 0/1), promised_days, same_state (int, 1 nếu MỌI seller của đơn
    cùng bang với khách), log_freight (log1p tổng freight_value), n_items, n_sellers, customer_state,
    handling_days (giao cho vận chuyển − mua), remaining_days (ngày hẹn − giao cho vận chuyển).
    Mọi khoảng thời gian tính bằng ngày, số thực. Đơn thiếu ngày giao vận chuyển để NaN.
    """
    # TODO: lọc đơn giao xong, gắn bang khách
    # TODO: gộp item theo đơn (số item, số seller, tổng freight, cùng bang)
    # TODO: tính các cột thời gian
    raise NotImplementedError


In [ ]:
# explore
df = build_orders(orders, items, customers, sellers)
print(len(df), df.late.mean())
df.describe()

## 2. Chia theo thời gian (định nghĩa chung của track)

In [ ]:
def time_split(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """(train, val, test) theo purchase_ts: < 2018-01-01 | [2018-01-01, 2018-04-01) | [2018-04-01, 2018-09-01)."""
    # TODO
    raise NotImplementedError


In [ ]:
# explore: prevalence khác nhau giữa ba tập (khủng hoảng 02–03/2018 nằm trong validation)
train, val, test = time_split(df)
for name, d in [("train", train), ("val", val), ("test", test)]:
    print(name, len(d), round(d.late.mean(), 4))

## 3. Feature hợp lệ theo thời điểm quyết định

In [ ]:
def features_at(decision_time: str) -> list[str]:
    """Các cột feature hợp lệ tại thời điểm quyết định: "purchase" (khách vừa đặt) hoặc "carrier" (đơn vừa
    giao cho vận chuyển). Thời điểm khác thì ValueError.
    """
    # TODO: chỉ dùng thông tin đã có tại thời điểm đó
    raise NotImplementedError


## 4. Metric offline và giá trị tiền

In [ ]:
def offline_report(y, p, ks=(100, 500, 1000)) -> dict[str, float]:
    """prevalence, roc_auc, pr_auc (average precision) và precision@k cho mỗi k (khoá "precision@<k>")."""
    # TODO
    raise NotImplementedError


In [ ]:
def value_table(y, p, fractions, cost_action: float, loss_event: float, effect: float,
                n_weeks: int) -> pd.DataFrame:
    """Mỗi tỉ lệ f trong fractions: chọn top k = floor(f * n) theo p. Index fraction; cột k, precision,
    value = TP * effect * loss_event − k * cost_action (R$ trên cả tập), value_per_week = value / n_weeks.
    """
    # TODO
    raise NotImplementedError


## 5. Model

In [ ]:
def fit_predict(train: pd.DataFrame, test: pd.DataFrame, features: list[str]) -> np.ndarray:
    """Train một classifier dự đoán `late` trên train[features], trả xác suất cho test (mảng 1 chiều).
    Phải xử lý được NaN (handling_days) và cột chữ customer_state. Model nào cũng được.
    """
    # TODO
    raise NotImplementedError


In [ ]:
# explore: so hai thời điểm trên test (train trên train)
preds = {}
for t in ["purchase", "carrier"]:
    preds[t] = fit_predict(train, test, features_at(t))
    print(t, {k: round(v, 3) for k, v in offline_report(test.late, preds[t]).items()})

In [ ]:
# explore: đổi ra tiền. Giả định: tin nhắn 0.5 R$, review xấu thiệt 30 R$, nhắn tin giảm 15 điểm % xác suất review xấu
weeks = test.purchase_ts.dt.to_period("W").nunique()
for t, p in preds.items():
    print(t)
    display(value_table(test.late, p, [0.01, 0.02, 0.05, 0.10, 0.20], 0.5, 30.0, 0.15, weeks).round(3))

In [ ]:
# explore: baseline không ML. Luật "còn dưới r ngày tới ngày hẹn lúc giao vận chuyển"
for r in [0, 3, 5, 7]:
    rule = (test.remaining_days < r).to_numpy()
    tp = test.late[rule].sum()
    print(r, rule.sum(), round(tp / rule.sum(), 3), round(tp / test.late.sum(), 3),
          round((tp * 0.15 * 30 - rule.sum() * 0.5) / weeks, 1), "R$/tuần")

## 6. Câu hỏi

1. PR-AUC của model lúc đặt hàng trên validation và test khác nhau bao nhiêu? Bao nhiêu phần do model,
   bao nhiêu do prevalence? (gợi ý: chia PR-AUC cho prevalence).
2. Đợi tới lúc giao vận chuyển đổi được gì, mất gì? Còn bao nhiêu ngày để hành động?
3. Vì sao precision@100 của model lúc giao vận chuyển gần 1? Đó có phải "model giỏi" không?
4. Luật một dòng so với model thế nào về giá trị mỗi tuần? Bạn sẽ ra mắt cái nào trước, vì sao?
5. Giá trị mỗi tuần có đáng xây hệ thống không? Giả định nào bạn cần business xác nhận?